In [ ]:
import pandas as pd
import numpy as np
import warnings
import empyrical
import dai
import bigcharts 
import time 
warnings.filterwarnings('ignore')
print('导入包完成!')

In [ ]:
sql = f"""
    with t_ret as (
        SELECT
            date,
            instrument,
            -- 日间收益率
            (close / NULLIF(LAG(close, 1) OVER (PARTITION BY instrument ORDER BY date), 0)) - 1 AS interday_ret,
            -- 日内收益率
            (close / NULLIF(open, 0)) - 1 AS intraday_ret,
            -- 隔夜收益率
            (open / NULLIF(LAG(close, 1) OVER (PARTITION BY instrument ORDER BY date), 0)) - 1 AS overnight_ret,

            -- 换手率差
            turn -  NULLIF(LAG(turn, 1) OVER (PARTITION BY instrument ORDER BY date), 0) AS turn_diff
        FROM
            cn_stock_prefactors
    ),

    t_ret_mean_std as (
        SELECT
            date,
            instrument,
            --换手率差
            turn_diff,
            -- 日间20 均值
            NANAVG(interday_ret) OVER (
                PARTITION BY instrument 
                ORDER BY date 
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS interday_ret_mean,
            -- 日间20 std
            STDDEV(interday_ret) OVER (
                PARTITION BY instrument 
                ORDER BY date 
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS interday_ret_std,
            -- 日内20 mean
            NANAVG(intraday_ret) OVER (
                PARTITION BY instrument 
                ORDER BY date 
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS intraday_ret_mean,
            -- 日内20 std
            STDDEV(intraday_ret) OVER (
                PARTITION BY instrument 
                ORDER BY date 
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS intraday_ret_std,
            -- 隔夜20 mean
            NANAVG(overnight_ret) OVER (
                PARTITION BY instrument 
                ORDER BY date 
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS overnight_ret_mean,
            -- 日内20 std
            STDDEV(overnight_ret) OVER (
                PARTITION BY instrument 
                ORDER BY date 
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS overnight_ret_std
        FROM
            t_ret
    ),

    t_std_turn_market_mean as (
        SElECT
            date,
            instrument,
            -- 日间收益波动截面均值
            AVG(interday_ret_std) over(partition by date) as   interday_ret_std_market_mean,
            -- 日内收益波动截面均值
            AVG(intraday_ret_std) over(partition by date) as   intraday_ret_std_market_mean,
            -- 隔夜收益波动截面均值
            AVG(overnight_ret_std) over(partition by date) as  overnight_ret_std_market_mean,

            -- 换手率差截面均值
            AVG(turn_diff) over(partition by date) as  turn_diff_market_mean,
        FROM
            t_ret_mean_std
    ),
    
    t_coin_team as (
        SELECT 
            date,
            instrument,
            interday_ret_std,
            interday_ret_std_market_mean,
            intraday_ret_std,
            intraday_ret_std_market_mean,
            overnight_ret_std,
            overnight_ret_std_market_mean,
            turn_diff,
            turn_diff_market_mean,
            
            --- 日间coin_team 
            CASE
                WHEN (interday_ret_std - interday_ret_std_market_mean) > 0 THEN 1
                WHEN (interday_ret_std - interday_ret_std_market_mean) < 0 THEN -1
                ELSE 0
            END AS _interday_ret_signal,
            interday_ret_mean * _interday_ret_signal   as interday_std_coin_team,

            --- 日间coin_team
            CASE
                WHEN (intraday_ret_std - intraday_ret_std_market_mean) > 0 THEN 1
                WHEN (intraday_ret_std - intraday_ret_std_market_mean) < 0 THEN -1
                ELSE 0
            END AS _intraday_ret_signal,
            intraday_ret_mean * _intraday_ret_signal  as intraday_std_coin_team,

            --- 隔夜coin_team
            CASE
                WHEN (overnight_ret_std - overnight_ret_std_market_mean) > 0 THEN 1
                WHEN (overnight_ret_std - overnight_ret_std_market_mean) < 0 THEN -1
                ELSE 0
            END AS _overnight_ret_signal,
            overnight_ret_mean * _overnight_ret_signal  as overnight_std_coin_team,

            --换手率差coin_team
            CASE
                WHEN (turn_diff - turn_diff_market_mean) > 0 THEN 1
                WHEN (turn_diff - turn_diff_market_mean) < 0 THEN -1
                ELSE 0
            END AS _turn_diff_signal,

            --日间-换手coin_team_raw
            interday_ret_mean * _turn_diff_signal  as  interday_turn_coin_team_raw,
            --日内-换手coin_team_raw
            intraday_ret_mean * _turn_diff_signal  as intraday_turn_coin_team_raw,
            --隔夜-换手coin_team_raw
            overnight_ret_mean * _turn_diff_signal  as overnight_turn_coin_team_raw

        FROM
            t_ret_mean_std
        JOIN
            t_std_turn_market_mean USING(date,instrument)
    ),

    t_coin_team_final as (
        SELECT
            date,
            instrument,
            interday_std_coin_team,
            intraday_std_coin_team,
            overnight_std_coin_team,
            --日间-换手coin_team
            NANAVG(interday_turn_coin_team_raw) OVER (
                PARTITION BY instrument 
                ORDER BY date 
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS interday_turn_coin_team,

            --日内-换手coin_team
            NANAVG(intraday_turn_coin_team_raw) OVER (
                PARTITION BY instrument 
                ORDER BY date 
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS intraday_turn_coin_team,

            --隔夜-换手coin_team
            NANAVG(overnight_turn_coin_team_raw) OVER (
                PARTITION BY instrument 
                ORDER BY date 
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS overnight_turn_coin_team
        FROM
            t_coin_team
    )
    

    SELECT
        date,
        instrument,
        --日内修正coin_team
        (interday_std_coin_team  +  interday_turn_coin_team)/2 as revise_interday_coin_team,
        --日间修正coin_team
        (intraday_std_coin_team  +  intraday_turn_coin_team)/2 as revise_intraday_coin_team,
        --隔夜修正coint_team
        (overnight_std_coin_team +  overnight_turn_coin_team)/2 as revise_overnight_coin_team,

        -- coin_team
        c_zscore(revise_interday_coin_team) + c_zscore(revise_intraday_coin_team) + c_zscore(revise_overnight_coin_team) as factor

        
    FROM
        t_coin_team_final
    
"""

In [ ]:
def run(sql,shift_days):

    from bigquant import bigtrader, dai
    import pandas as pd
    from datetime import datetime, timedelta
    import numpy as np
    from sklearn.linear_model import LinearRegression

    def initialize(context: bigtrader.IContext):
        from bigtrader.finance.commission import PerOrder

        # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
        context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))


        context.holding_days = 5
        context.target_hold_count = 50


    def befor_trading(context, data):
        pass
        
    def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):

        # 每 context.holding_days 个交易日调仓一次
        if context.trading_day_index % context.holding_days != 0:
            return




        # 获取当前日期
        ed = data.current_dt.strftime("%Y-%m-%d")
        
        date_obj = datetime.strptime(ed, "%Y-%m-%d")
    
        # 向前推10天
        n_days_ago = date_obj - timedelta(days=shift_days)
        tomorrow = date_obj + timedelta(days=1)
    
        # 转换回字符串格式
        sd = n_days_ago.strftime("%Y-%m-%d")
        ed2 = tomorrow.strftime("%Y-%m-%d")
        
        # 获取当日数据
        current_day_data = dai.query(sql,filters={'date':[sd,ed2]}).df()
        current_day_data['date']=pd.to_datetime(current_day_data['date']) 
        current_day_data = current_day_data[current_day_data.date==ed]

        # 取前10只
        current_day_data.sort_values(by='factor',inplace=True,ascending=True)

        current_day_data = current_day_data.head(context.target_hold_count)
        len_ = len(current_day_data)
        # 获取当日目标持有股票
        target_hold_instruments = set(current_day_data["instrument"])
        
        # 获取当前已持有股票
        current_hold_instruments = set(context.get_account_positions().keys())

        # 卖出不在目标持有列表中的股票
        for instrument in current_hold_instruments - target_hold_instruments:
            context.order_target_percent(instrument, 0)
            
        # 买入目标持有列表中的股票
        for instrument in target_hold_instruments - current_hold_instruments:
            context.order_target_percent(instrument, 1/len_)

    performance = bigtrader.run(
        market=bigtrader.Market.CN_STOCK,
        frequency=bigtrader.Frequency.DAILY,
        start_date='2021-12-31',  
        end_date='2025-12-31',  
        capital_base=3000000,     # 设置初始资金
        initialize=initialize,     # 传入初始化函数
        handle_data=handle_data,   # 传入数据处理函数
        before_trading_start = befor_trading,
        order_price_field_buy='open',
        order_price_field_sell='open',
    )

    # 渲染绩效报告，展示回测结果
    performance.render()

In [ ]:
run(sql,21)